<a href="https://colab.research.google.com/github/JasonSupala/DataMining/blob/main/FinalCodingDataMining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install shap

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import xgboost as xgb

In [ ]:
df = pd.read_csv("obesity_train.csv")
df.head()

,Age,Gender,Height,Weight,CALC,FAVC,FCVC,NCP,SCC,SMOKE,CH2O,family_history_with_overweight,FAF,TUE,CAEC,MTRANS,NObeyesdad
0,23.000000,Female,1.644161,84.340406,no,yes,2.177243,3.000000,no,no,2.715572,yes,2.230109,0.070897,Sometimes,Public_Transportation,Obesity_Type_I
1,22.000000,Male,1.650000,80.000000,no,no,2.000000,3.000000,no,no,2.000000,yes,3.000000,2.000000,Sometimes,Walking,Overweight_Level_II
2,20.670975,Female,1.509408,64.852953,no,no,2.294067,3.209508,no,no,2.000000,yes,1.103088,1.261043,Sometimes,Public_Transportation,Overweight_Level_II
3,23.000000,Male,1.706525,90.500055,no,yes,2.000000,3.000000,no,no,1.530493,yes,0.967627,1.000000,Sometimes,Public_Transportation,Obesity_Type_I
4,27.000000,Male,1.800000,87.000000,Frequently,no,3.000000,3.000000,no,no,2.000000,no,2.000000,0.000000,Sometimes,Walking,Overweight_Level_I


In [ ]:
KnownClass = list(df['NObeyesdad'].unique())
CategoryCol = list(df.columns)
print(KnownClass)
print(CategoryCol)

['Obesity_Type_I', 'Overweight_Level_II', 'Overweight_Level_I', 'Insufficient_Weight', 'Normal_Weight']
['Age', 'Gender', 'Height', 'Weight', 'CALC', 'FAVC', 'FCVC', 'NCP', 'SCC', 'SMOKE', 'CH2O', 'family_history_with_overweight', 'FAF', 'TUE', 'CAEC', 'MTRANS', 'NObeyesdad']


In [ ]:
dfT = pd.read_csv("obesity_test.csv")

In [ ]:
print(list(dfT.columns))
print(dfT['NObeyesdad'].unique())

['Age', 'Gender', 'Height', 'Weight', 'CALC', 'FAVC', 'FCVC', 'NCP', 'SCC', 'SMOKE', 'CH2O', 'family_history_with_overweight', 'FAF', 'TUE', 'CAEC', 'MTRANS', 'NObeyesdad']
['Obesity_Type_II' 'Obesity_Type_III' 'Normal_Weight' 'Overweight_Level_I'
 'Insufficient_Weight' 'Obesity_Type_I' 'Overweight_Level_II']


In [ ]:
UnknownClass = list(set(dfT['NObeyesdad'].unique()).difference(KnownClass))
print(UnknownClass)

['Obesity_Type_II', 'Obesity_Type_III']


The code starts below

Preprocessing

In [ ]:
#We will now preprocess our data by turn each data into 'category' for the xgboost, because XGBoost’s native categorical
#handling only works when the column is recognized as categorical by pandas

def preprocess(df):
    df = df.copy()
    for col in CategoryCol:
        if col in df.columns:
            df[col] = df[col].astype(str).astype("category")
    return df

BMI function and some facts

In [ ]:
def ComputeBMI(df):
    return df["Weight"] / (df["Height"] ** 2)


ObeseType2BMI_MIN  = 35.0
ObeseType3BMI_MIN = 40.0

Clustering Functions

In [ ]:
def cluster_unlabeled(X_unlabeled, features=("Weight", "Height"), n_clusters=2):
    X = X_unlabeled.loc[:, features].to_numpy()
    X_scaled = StandardScaler().fit_transform(X)
    return KMeans(n_clusters=n_clusters, n_init="auto", random_state=42,).fit_predict(X_scaled)

In [ ]:
def map_clusters_to_labels(X_unlabeled, cluster_labels):
    X_unlabeled = X_unlabeled.copy().reset_index(drop=True)
    X_unlabeled["BMI"]     = ComputeBMI(X_unlabeled)
    X_unlabeled["cluster"] = cluster_labels
    final_labels = []
    for _, row in X_unlabeled.iterrows():
        if row["BMI"] >= ObeseType3BMI_MIN:
            final_labels.append("Obesity_Type_III")
        elif row["BMI"] >= ObeseType2BMI_MIN and row["BMI"] < ObeseType3BMI_MIN:
            final_labels.append("Obesity_Type_II")

    return np.array(final_labels)

Build Classifier

In [ ]:
Cls = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,   # native categorical support
        tree_method="hist",      # required for enable_categorical
        eval_metric="mlogloss",
        verbosity=0,
        random_state=42,
    )

Using shap to check the importance of each feature from the trained model

In [ ]:
Clf = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,   # native categorical support
        tree_method="hist",      # required for enable_categorical
        eval_metric="mlogloss",
        verbosity=0,
        random_state=42,
    )
train_data = preprocess(df.drop(columns=['NObeyesdad']))

target_le = LabelEncoder()
y_train   = target_le.fit_transform(df['NObeyesdad'])
Clf.fit(train_data, y_train)


explainer = shap.TreeExplainer(Clf)
shap_values = explainer.shap_values(train_data)

# Convert to numpy array
shap_arr = np.array(shap_values)
print("Shape of shap array: ",{shap_arr.shape})

mean_shap = np.abs(shap_arr).mean(axis=(0, 2))

importance_df = pd.DataFrame({
    'Feature': train_data.columns,
    'Mean |SHAP|': mean_shap
}).sort_values('Mean |SHAP|', ascending=False)

print(importance_df)

Shape of shap array:  {(1193, 16, 5)}
                           Feature  Mean |SHAP|
3                           Weight     0.575411
0                              Age     0.421884
12                             FAF     0.361213
2                           Height     0.342167
13                             TUE     0.338107
11  family_history_with_overweight     0.337734
10                            CH2O     0.296938
7                              NCP     0.274401
14                            CAEC     0.252882
6                             FCVC     0.206657
1                           Gender     0.185627
4                             CALC     0.183922
5                             FAVC     0.133729
15                          MTRANS     0.095491
8                              SCC     0.029732
9                            SMOKE     0.001469


The pipeline

In [ ]:
def run_pipeline(threshold=0.90):
    df = pd.read_csv("obesity_train.csv")
    train_data = preprocess(df.drop(columns=['NObeyesdad']))

    target_le = LabelEncoder()
    y_train   = target_le.fit_transform(df['NObeyesdad'])
    Cls.fit(train_data, y_train)

    test_df     = pd.read_csv("obesity_test.csv")
    has_labels  = 'NObeyesdad' in test_df.columns
    X_test     = preprocess(test_df.drop(columns=['NObeyesdad']))
    TestCopy = test_df.copy()

    proba          = Cls.predict_proba(X_test)
    max_proba      = proba.max(axis=1)
    pred_labels    = target_le.inverse_transform(proba.argmax(axis=1))
    confident_mask = max_proba >= threshold
    uncertain_mask = ~confident_mask
    final_preds    = pred_labels.copy().astype(object)

    if uncertain_mask.sum() >= 2:
        uncertain_idx  = np.where(uncertain_mask)[0]
        X_uncertain    = TestCopy.iloc[uncertain_idx].reset_index(drop=True)
        cluster_labels = cluster_unlabeled(X_uncertain)
        cluster_named  = map_clusters_to_labels(X_uncertain, cluster_labels)
        for idx, label in zip(uncertain_idx, cluster_named):
            final_preds[idx] = label

    true_labels = test_df['NObeyesdad'].values
    known_mask  = np.isin(true_labels, KnownClass)
    print(f"Accuracy: {accuracy_score(true_labels[known_mask], final_preds[known_mask]):.4f}")
    print(classification_report(
        true_labels[known_mask], final_preds[known_mask],
        labels=sorted(KnownClass), target_names=sorted(KnownClass), zero_division=0,
    ))

    output_df = TestCopy.copy()
    output_df["BMI"]             = ComputeBMI(TestCopy)
    output_df["predicted_label"] = final_preds
    output_df["confidence"]      = max_proba
    output_df["source"]          = np.where(confident_mask, "classifier", "clustering")
    output_df.to_csv("obesity_predictions.csv", index=False)
    print("Saved → obesity_predictions.csv")
    return output_df

In [ ]:
run_pipeline()

XGBoostError: [19:12:49] /__w/xgboost/xgboost/src/data/../data/cat_container.h:29: Found a category not in the training set for the 0th (0-based) column: `61.0`
Stack trace:
  [bt] (0) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x2c1a8c) [0x7ff83d2c1a8c]
  [bt] (1) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x62c020) [0x7ff83d62c020]
  [bt] (2) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x62c0c1) [0x7ff83d62c0c1]
  [bt] (3) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x64bf02) [0x7ff83d64bf02]
  [bt] (4) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x8f46a3) [0x7ff83d8f46a3]
  [bt] (5) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x9274ea) [0x7ff83d9274ea]
  [bt] (6) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x927c84) [0x7ff83d927c84]
  [bt] (7) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x927f6b) [0x